In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
import json
import time
from datetime import datetime
from scipy.io import loadmat
from scipy.special import logsumexp
import scipy.linalg
from sklearn.mixture import GaussianMixture
import gc
import warnings
warnings.filterwarnings('ignore')


class TimeManager:
    """Manage training time limits for any environment"""
    
    def __init__(self, max_hours=240):
        self.start_time = time.time()
        self.max_seconds = max_hours * 3600
        self.last_checkpoint = self.start_time
        
    def elapsed_hours(self):
        """Get elapsed time in hours"""
        return (time.time() - self.start_time) / 3600
    
    def elapsed_seconds(self):
        """Get elapsed time in seconds"""
        return time.time() - self.start_time
    
    def time_remaining_hours(self):
        """Get remaining time in hours"""
        return max(0, self.max_seconds/3600 - self.elapsed_hours())
    
    def should_stop(self, buffer_minutes=30):
        """Check if we should stop training (with buffer for saving)"""
        buffer_seconds = buffer_minutes * 60
        return (time.time() - self.start_time + buffer_seconds) >= self.max_seconds
    
    def time_for_checkpoint(self, interval_minutes=30):
        """Check if it's time for a checkpoint"""
        if (time.time() - self.last_checkpoint) >= (interval_minutes * 60):
            self.last_checkpoint = time.time()
            return True
        return False
    
    def format_elapsed_time(self):
        """Format elapsed time as HH:MM:SS"""
        elapsed = int(self.elapsed_seconds())
        hours = elapsed // 3600
        minutes = (elapsed % 3600) // 60
        seconds = elapsed % 60
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    
    def estimate_remaining_items(self, items_completed, total_items):
        """Estimate how many more items can be completed"""
        if items_completed == 0:
            return total_items
        
        avg_time_per_item = self.elapsed_seconds() / items_completed
        remaining_time = self.time_remaining_hours() * 3600 - 1800
        
        return max(0, int(remaining_time / avg_time_per_item))


class MultivariateGaussianMixtureHMM:
    """
    Multivariate HMM with GMM emissions (full covariances).
    Tracks convergence criterion: log_likelihood, parameter_stationarity, 
    stationary_flow, multiple_criteria, or max_iterations.
    """
    def __init__(self,
                 n_states=3,
                 n_gmm_components=3,
                 max_iter=50,
                 tol=1e-3,
                 random_state=42,
                 param_tol=1e-3,
                 min_param_iters=2,
                 early_stop_by_params=True,
                 flow_tol=1e-3,
                 flow_norm='inf',
                 stop_on_flow=True):
        self.n_states = n_states
        self.n_gmm_components = n_gmm_components
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

        # Model parameters
        self.pi = None
        self.A = None
        self.gmm_weights = None
        self.gmm_means = None
        self.gmm_covars = None

        # Training state
        self.log_likelihoods = []
        self.converged = False
        self.final_iteration = 0
        self.n_features = None
        self.convergence_criterion = None

        # Stationary distribution cache
        self.stationary_pi = None

        # Stationarity controls
        self.param_tol = param_tol
        self.min_param_iters = min_param_iters
        self.early_stop_by_params = early_stop_by_params

        self.flow_tol = flow_tol
        self.flow_norm = flow_norm
        self.stop_on_flow = stop_on_flow
        self._prev_A_flow = None

    def _initialize_parameters(self, X):
        np.random.seed(self.random_state)
        T, n_features = X.shape
        self.n_features = n_features

        self.pi = np.ones(self.n_states) / self.n_states
        self.A = np.random.dirichlet(np.ones(self.n_states), self.n_states)

        self.gmm_weights = np.zeros((self.n_states, self.n_gmm_components))
        self.gmm_means   = np.zeros((self.n_states, self.n_gmm_components, n_features))
        self.gmm_covars  = np.zeros((self.n_states, self.n_gmm_components, n_features, n_features))

        segment_size = max(1, T // self.n_states)

        for s in range(self.n_states):
            start_idx = s * segment_size
            end_idx = (s + 1) * segment_size if s < self.n_states - 1 else T
            segment_data = X[start_idx:end_idx]

            if len(segment_data) < self.n_gmm_components * 2:
                self.gmm_weights[s] = np.ones(self.n_gmm_components) / self.n_gmm_components
                segment_mean = np.mean(segment_data, axis=0)
                segment_cov  = np.cov(segment_data.T) + np.eye(n_features) * 1e-6
                for k in range(self.n_gmm_components):
                    jitter = np.random.multivariate_normal(np.zeros(n_features), 0.1 * segment_cov)
                    self.gmm_means[s, k] = segment_mean + jitter
                    self.gmm_covars[s, k] = segment_cov
            else:
                try:
                    gmm = GaussianMixture(n_components=self.n_gmm_components,
                                          random_state=self.random_state + s,
                                          covariance_type='full',
                                          reg_covar=1e-6)
                    gmm.fit(segment_data)
                    self.gmm_weights[s] = gmm.weights_
                    self.gmm_means[s]   = gmm.means_
                    self.gmm_covars[s]  = gmm.covariances_
                except Exception:
                    self.gmm_weights[s] = np.ones(self.n_gmm_components) / self.n_gmm_components
                    segment_mean = np.mean(segment_data, axis=0)
                    segment_cov  = np.cov(segment_data.T) + np.eye(n_features) * 1e-6
                    for k in range(self.n_gmm_components):
                        self.gmm_means[s, k]  = segment_mean
                        self.gmm_covars[s, k] = segment_cov

            for k in range(self.n_gmm_components):
                self.gmm_covars[s, k] += np.eye(n_features) * 1e-6

    def _multivariate_log_likelihood(self, x, mean, cov):
        d = len(x)
        cov_reg = cov + np.eye(d) * 1e-6
        try:
            L = np.linalg.cholesky(cov_reg)
            log_det = 2 * np.sum(np.log(np.diag(L)))
            diff = x - mean
            y = scipy.linalg.solve_triangular(L, diff, lower=True)
            maha = np.dot(y, y)
            return -0.5 * (d * np.log(2 * np.pi) + log_det + maha)
        except np.linalg.LinAlgError:
            cov_diag = np.diag(np.diag(cov_reg))
            log_det = np.sum(np.log(np.diag(cov_diag)))
            diff = x - mean
            maha = np.sum((diff ** 2) / np.diag(cov_diag))
            return -0.5 * (d * np.log(2 * np.pi) + log_det + maha)

    def _compute_log_emission_probs(self, X):
        T, _ = X.shape
        log_probs = np.zeros((T, self.n_states))
        for t in range(T):
            xt = X[t]
            for s in range(self.n_states):
                comps = np.zeros(self.n_gmm_components)
                for k in range(self.n_gmm_components):
                    comps[k] = (
                        np.log(self.gmm_weights[s, k] + 1e-12) +
                        self._multivariate_log_likelihood(xt, self.gmm_means[s, k], self.gmm_covars[s, k])
                    )
                log_probs[t, s] = logsumexp(comps)
        return log_probs

    def _forward_pass(self, log_emission_probs):
        T, K = log_emission_probs.shape
        log_alpha = np.zeros((T, K))
        log_scales = np.zeros(T)

        log_alpha[0] = np.log(self.pi + 1e-12) + log_emission_probs[0]
        log_scales[0] = logsumexp(log_alpha[0])
        log_alpha[0] -= log_scales[0]

        for t in range(1, T):
            for j in range(K):
                log_alpha[t, j] = logsumexp(log_alpha[t-1] + np.log(self.A[:, j] + 1e-12)) + log_emission_probs[t, j]
            log_scales[t] = logsumexp(log_alpha[t])
            log_alpha[t] -= log_scales[t]
        return log_alpha, log_scales

    def _backward_pass(self, log_emission_probs, log_scales):
        T, K = log_emission_probs.shape
        log_beta = np.zeros((T, K))
        log_beta[T-1] = -log_scales[T-1]
        for t in range(T-2, -1, -1):
            for i in range(K):
                log_beta[t, i] = logsumexp(
                    np.log(self.A[i, :] + 1e-12) + log_emission_probs[t+1] + log_beta[t+1]
                ) - log_scales[t]
        return log_beta

    def _compute_posteriors(self, log_alpha, log_beta, log_emission_probs):
        T, K = log_alpha.shape
        log_gamma = log_alpha + log_beta
        gamma = np.exp(log_gamma)
        gamma = gamma / (gamma.sum(axis=1, keepdims=True) + 1e-12)

        xi = np.zeros((T-1, K, K))
        for t in range(T-1):
            for i in range(K):
                for j in range(K):
                    xi[t, i, j] = np.exp(
                        log_alpha[t, i] + np.log(self.A[i, j] + 1e-12) +
                        log_emission_probs[t+1, j] + log_beta[t+1, j]
                    )
            s = xi[t].sum()
            if s > 0:
                xi[t] /= s
        return gamma, xi

    def _compute_gmm_posteriors(self, X, gamma):
        T, _ = X.shape
        gmm_gamma = np.zeros((T, self.n_states, self.n_gmm_components))
        for t in range(T):
            xt = X[t]
            for s in range(self.n_states):
                comps = np.zeros(self.n_gmm_components)
                for k in range(self.n_gmm_components):
                    comps[k] = (
                        np.log(self.gmm_weights[s, k] + 1e-12) +
                        self._multivariate_log_likelihood(xt, self.gmm_means[s, k], self.gmm_covars[s, k])
                    )
                comps -= logsumexp(comps)
                gmm_gamma[t, s] = np.exp(comps)
        return gmm_gamma

    def _update_parameters(self, X, gamma, xi):
        T, D = X.shape

        self.pi = gamma[0]

        xi_sum = xi.sum(axis=0)
        gamma_sum = gamma[:-1].sum(axis=0)
        for i in range(self.n_states):
            if gamma_sum[i] > 1e-12:
                self.A[i, :] = xi_sum[i, :] / gamma_sum[i]
            else:
                self.A[i, :] = 1.0 / self.n_states
        self.A = self.A / (self.A.sum(axis=1, keepdims=True) + 1e-12)

        gmm_gamma = self._compute_gmm_posteriors(X, gamma)
        for s in range(self.n_states):
            gamma_s_sum = gamma[:, s].sum()
            if gamma_s_sum <= 1e-12:
                continue
            for k in range(self.n_gmm_components):
                w = gamma[:, s] * gmm_gamma[:, s, k]
                wsum = w.sum()
                if wsum > 1e-12:
                    self.gmm_weights[s, k] = wsum / gamma_s_sum
                    self.gmm_means[s, k] = (w[:, None] * X).sum(axis=0) / wsum
                    diff = X - self.gmm_means[s, k]
                    self.gmm_covars[s, k] = (diff.T * w) @ diff / wsum
                    self.gmm_covars[s, k] += np.eye(D) * 1e-6
            self.gmm_weights[s] /= (self.gmm_weights[s].sum() + 1e-12)

    def _snapshot_params(self):
        return {
            'pi': self.pi.copy(),
            'A': self.A.copy(),
            'weights': self.gmm_weights.copy(),
            'means': self.gmm_means.copy(),
            'covs': self.gmm_covars.copy()
        }

    def _param_max_delta(self, prev, curr):
        deltas = {
            'pi':    np.max(np.abs(curr['pi']      - prev['pi'])),
            'A':     np.max(np.abs(curr['A']       - prev['A'])),
            'w':     np.max(np.abs(curr['weights'] - prev['weights'])),
            'mu':    np.max(np.abs(curr['means']   - prev['means'])),
            'Sigma': np.max(np.abs(curr['covs']    - prev['covs'])),
        }
        return max(deltas.values()), deltas

    def _stationary_from_A(self, A, rel_tol=1e-8, abs_tol=1e-10, max_iter=2000, eps=1e-12, damping=0.0):
        """Compute stationary distribution from transition matrix"""
        A = np.asarray(A, dtype=float)
        rs = A.sum(axis=1, keepdims=True)
        rs[rs == 0] = 1.0
        A = A / rs
        K = A.shape[0]

        if damping > 0.0:
            J = np.ones((K, K), dtype=float) / K
            A = (1.0 - damping) * A + damping * J

        pi = np.ones(K, dtype=float) / K
        for _ in range(max_iter):
            pi_new = pi @ A
            if np.linalg.norm(pi_new - pi, 1) <= rel_tol * np.linalg.norm(pi, 1) + abs_tol:
                pi = pi_new
                break
            pi = pi_new

        pi = np.maximum(pi, 0.0)
        s = pi.sum()
        return pi / (s if s > 0 else (1.0 + eps))

    def _stationary_flow_matrix(self, A):
        pi_s = self._stationary_from_A(A)
        A_flow = pi_s[:, None] * A
        return A_flow, pi_s

    def fit(self, X, verbose=False):
        X = np.array(X, dtype=np.float64)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        self._initialize_parameters(X)
        prev_log_likelihood = -np.inf
        prev_params = self._snapshot_params()

        for iteration in range(self.max_iter):
            try:
                # E-step
                log_emission_probs = self._compute_log_emission_probs(X)
                log_alpha, log_scales = self._forward_pass(log_emission_probs)
                log_beta = self._backward_pass(log_emission_probs, log_scales)
                gamma, xi = self._compute_posteriors(log_alpha, log_beta, log_emission_probs)

                # Log likelihood
                log_likelihood = log_scales.sum()
                self.log_likelihoods.append(log_likelihood)
                if verbose and iteration % 5 == 0:
                    print(f"    Iter {iteration}: LL={log_likelihood:.6f}")

                # M-step
                self._update_parameters(X, gamma, xi)

                # Track which criteria are satisfied
                criteria_met = []

                # Check parameter stationarity
                param_ok = False
                if self.early_stop_by_params and iteration >= self.min_param_iters:
                    curr_params = self._snapshot_params()
                    overall_delta, block_deltas = self._param_max_delta(prev_params, curr_params)
                    if verbose:
                        print("      Param delta (inf-norm):",
                              f"π={block_deltas['pi']:.3e}, A={block_deltas['A']:.3e},",
                              f"w={block_deltas['w']:.3e}, mu={block_deltas['mu']:.3e}, Sigma={block_deltas['Sigma']:.3e},",
                              f"overall={overall_delta:.3e}")
                    param_ok = (overall_delta < self.param_tol)
                    if param_ok:
                        criteria_met.append('parameter_stationarity')
                    prev_params = curr_params

                # Check stationary flow
                flow_ok = False
                if self.stop_on_flow:
                    A_flow, pi_s_now = self._stationary_flow_matrix(self.A)
                    if self._prev_A_flow is not None:
                        diff = A_flow - self._prev_A_flow
                        if self.flow_norm == 'fro':
                            flow_delta = np.linalg.norm(diff, ord='fro')
                        elif self.flow_norm == 'inf':
                            flow_delta = np.max(np.abs(diff))
                        else:
                            flow_delta = np.linalg.norm(diff)
                        if verbose:
                            print(f"      ||diag(pi_s)A|| delta = {flow_delta:.3e} ({self.flow_norm})")
                        flow_ok = (flow_delta < self.flow_tol)
                        if flow_ok:
                            criteria_met.append('stationary_flow')
                    self._prev_A_flow = A_flow
                    self.stationary_pi = pi_s_now

                # Check log-likelihood convergence
                ll_ok = False
                if abs(log_likelihood - prev_log_likelihood) < self.tol:
                    ll_ok = True
                    criteria_met.append('log_likelihood')

                # Decide on early stop
                if criteria_met and iteration >= self.min_param_iters:
                    self.converged = True
                    self.final_iteration = iteration
                    if len(criteria_met) > 1:
                        self.convergence_criterion = 'multiple_criteria'
                    else:
                        self.convergence_criterion = criteria_met[0]
                    if verbose:
                        print(f"    Converged at iter {iteration}: {self.convergence_criterion}")
                    break

                prev_log_likelihood = log_likelihood

            except Exception as e:
                if verbose:
                    print(f"    Error at iter {iteration}: {e}")
                break

        # If we exited without converging
        if not self.converged:
            self.convergence_criterion = 'max_iterations'
            self.final_iteration = iteration

        # Cache final stationary pi
        try:
            self.stationary_pi = self._stationary_from_A(self.A)
        except Exception:
            self.stationary_pi = None

        return self

    def get_parameters_dict(self):
        params = {}
        for i in range(self.n_states):
            params[f'pi_{i}'] = self.pi[i]
        for i in range(self.n_states):
            for j in range(self.n_states):
                params[f'A_{i}{j}'] = self.A[i, j]
        for s in range(self.n_states):
            for k in range(self.n_gmm_components):
                params[f'gmm_weight_{s}_{k}'] = self.gmm_weights[s, k]
                for f in range(self.n_features):
                    params[f'gmm_mean_{s}_{k}_f{f}'] = self.gmm_means[s, k, f]
                cov = self.gmm_covars[s, k]
                for i in range(self.n_features):
                    for j in range(i, self.n_features):
                        params[f'gmm_cov_{s}_{k}_f{i}f{j}'] = cov[i, j]
        params['final_log_likelihood'] = self.log_likelihoods[-1] if self.log_likelihoods else np.nan
        params['converged'] = self.converged
        params['final_iteration'] = self.final_iteration
        params['convergence_criterion'] = self.convergence_criterion
        params['n_states'] = self.n_states
        params['n_gmm_components'] = self.n_gmm_components
        params['n_features'] = self.n_features
        if self.stationary_pi is not None:
            for i, p in enumerate(self.stationary_pi):
                params[f'stationary_pi_{i}'] = p
        return params


class HMMTrainingTracker:
    """Track training progress and manage checkpoints"""
    
    def __init__(self, checkpoint_dir='./hmm_checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        self.progress_file = os.path.join(checkpoint_dir, 'training_progress.json')
        
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        self.progress = self.load_progress()
        self.training_times = []
    
    def load_progress(self):
        if os.path.exists(self.progress_file):
            with open(self.progress_file, 'r') as f:
                return json.load(f)
        else:
            return {
                'completed_hmms': [],
                'failed_hmms': [],
                'total_planned': 0,
                'start_time': None,
                'last_update': None,
                'training_times': []
            }
    
    def save_progress(self):
        self.progress['last_update'] = datetime.now().isoformat()
        with open(self.progress_file, 'w') as f:
            json.dump(self.progress, f, indent=2)
    
    def is_completed(self, series_id, dataset_type):
        hmm_key = f"{dataset_type}_{series_id}"
        completed_keys = [item['hmm_key'] for item in self.progress['completed_hmms']]
        return hmm_key in completed_keys
    
    def mark_completed(self, series_id, dataset_type, checkpoint_path, training_time=None):
        hmm_key = f"{dataset_type}_{series_id}"
        completion_info = {
            'hmm_key': hmm_key,
            'checkpoint_path': checkpoint_path,
            'completion_time': datetime.now().isoformat()
        }
        
        if training_time is not None:
            completion_info['training_time_seconds'] = training_time
            self.training_times.append(training_time)
            self.progress['training_times'] = self.training_times
        
        self.progress['completed_hmms'].append(completion_info)
        self.save_progress()
    
    def mark_failed(self, series_id, dataset_type, error_msg):
        hmm_key = f"{dataset_type}_{series_id}"
        self.progress['failed_hmms'].append({
            'hmm_key': hmm_key,
            'error': str(error_msg)[:200],
            'failure_time': datetime.now().isoformat()
        })
        self.save_progress()
    
    def get_remaining_work(self, total_adhd, total_control):
        completed_keys = [item['hmm_key'] for item in self.progress['completed_hmms']]
        failed_keys = [item['hmm_key'] for item in self.progress['failed_hmms']]
        
        remaining = []
        
        for series_id in range(total_adhd):
            hmm_key = f"ADHD_{series_id}"
            if hmm_key not in completed_keys and hmm_key not in failed_keys:
                remaining.append(('ADHD', series_id))
        
        for series_id in range(total_control):
            hmm_key = f"Control_{series_id}"
            if hmm_key not in completed_keys and hmm_key not in failed_keys:
                remaining.append(('Control', series_id))
        
        return remaining
    
    def print_status(self, time_manager=None):
        completed = len(self.progress['completed_hmms'])
        failed = len(self.progress['failed_hmms'])
        total = self.progress.get('total_planned', completed + failed)
        remaining = max(0, total - completed - failed)
        
        print(f"\n TRAINING PROGRESS")
        print(f"   Completed: {completed}/{total} ({100*completed/max(total,1):.1f}%)")
        print(f"   Failed: {failed}")
        print(f"   Remaining: {remaining}")
        
        if time_manager:
            print(f"   Elapsed time: {time_manager.format_elapsed_time()}")
            print(f"   Time remaining: {time_manager.time_remaining_hours():.1f}h")
            
            if self.training_times:
                avg_time = np.mean(self.training_times)
                est_remaining = time_manager.estimate_remaining_items(completed, total)
                print(f"   Avg time/HMM: {avg_time:.1f}s")
                print(f"   Can complete ~{est_remaining} more HMMs")
        
        print(f"   Last update: {self.progress.get('last_update', 'Never')}")


def load_eeg_mat_file(filepath):
    """
    Load EEG data from .mat file.
    Expected format: array with shape (n_samples, n_channels)
    - Rows: time samples
    - Columns: channels/features
    
    Parameters:
    -----------
    filepath : str
        Path to .mat file
        
    Returns:
    --------
    eeg_data : ndarray
        EEG data with shape (n_samples, n_channels)
    """
    try:
        mat_data = loadmat(filepath)
        
        # Try common variable names
        possible_keys = ['data', 'EEG', 'eeg_data', 'signal', 'X', 'channels', 'eeg']
        eeg_data = None
        
        for key in possible_keys:
            if key in mat_data:
                eeg_data = mat_data[key]
                break
        
        # If not found, look for the largest numeric array
        if eeg_data is None:
            numeric_keys = [k for k, v in mat_data.items()
                            if isinstance(v, np.ndarray) and v.ndim >= 1 and not k.startswith('__')]
            if numeric_keys:
                key = max(numeric_keys, key=lambda k: mat_data[k].size)
                eeg_data = mat_data[key]
        
        if eeg_data is None:
            raise ValueError("Could not find EEG data in .mat file")
        
        # Ensure it's 2D
        if eeg_data.ndim == 1:
            eeg_data = eeg_data.reshape(-1, 1)
        
        # Ensure shape is (n_samples, n_channels)
        # If n_channels > n_samples, transpose
        if eeg_data.shape[1] > eeg_data.shape[0]:
            eeg_data = eeg_data.T
        
        return eeg_data
        
    except Exception as e:
        raise ValueError(f"Error loading {filepath}: {str(e)}")


def load_all_eeg_data(adhd_dir, control_dir):
    """
    Load all EEG .mat files from directories.
    
    Parameters:
    -----------
    adhd_dir : str
        Directory containing ADHD .mat files
    control_dir : str
        Directory containing Control .mat files
        
    Returns:
    --------
    adhd_data : list of arrays
        ADHD time series
    control_data : list of arrays
        Control time series
    adhd_files : list
        ADHD filenames
    control_files : list
        Control filenames
    adhd_failed : list
        Failed ADHD files
    control_failed : list
        Failed Control files
    """
    print("  Loading EEG data from .mat files...")
    
    adhd_files = sorted([f for f in os.listdir(adhd_dir) if f.endswith('.mat')])
    control_files = sorted([f for f in os.listdir(control_dir) if f.endswith('.mat')])
    
    adhd_data, adhd_failed = [], []
    control_data, control_failed = [], []
    
    print(f"   Loading {len(adhd_files)} ADHD files from: {adhd_dir}")
    for i, filename in enumerate(adhd_files):
        try:
            filepath = os.path.join(adhd_dir, filename)
            eeg_data = load_eeg_mat_file(filepath)
            adhd_data.append(eeg_data)
            if i % 10 == 0:
                print(f"     Loaded ADHD {i+1}/{len(adhd_files)} - Shape: {eeg_data.shape}")
        except Exception as e:
            print(f"       Failed to load {filename}: {str(e)[:50]}")
            adhd_failed.append((filename, str(e)))
    
    print(f"   Loading {len(control_files)} Control files from: {control_dir}")
    for i, filename in enumerate(control_files):
        try:
            filepath = os.path.join(control_dir, filename)
            eeg_data = load_eeg_mat_file(filepath)
            control_data.append(eeg_data)
            if i % 10 == 0:
                print(f"     Loaded Control {i+1}/{len(control_files)} - Shape: {eeg_data.shape}")
        except Exception as e:
            print(f"       Failed to load {filename}: {str(e)[:50]}")
            control_failed.append((filename, str(e)))
    
    print(f"\n Data loading complete:")
    print(f"   ADHD: {len(adhd_data)} successful, {len(adhd_failed)} failed")
    if adhd_data:
        print(f"     First ADHD shape: {adhd_data[0].shape} (samples={adhd_data[0].shape[0]}, channels={adhd_data[0].shape[1]})")
    print(f"   Control: {len(control_data)} successful, {len(control_failed)} failed")
    if control_data:
        print(f"     First Control shape: {control_data[0].shape} (samples={control_data[0].shape[0]}, channels={control_data[0].shape[1]})")
    
    return adhd_data, control_data, adhd_files, control_files, adhd_failed, control_failed


def save_hmm_checkpoint(hmm, series_id, dataset_type, checkpoint_dir, filename_base):
    """Save individual HMM with metadata"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    checkpoint = {
        'hmm_object': hmm,
        'series_id': series_id,
        'dataset_type': dataset_type,
        'filename_base': filename_base,
        'timestamp': timestamp,
        'parameters': hmm.get_parameters_dict(),
        'convergence_info': {
            'converged': hmm.converged,
            'final_iteration': hmm.final_iteration,
            'convergence_criterion': hmm.convergence_criterion,
            'log_likelihoods': hmm.log_likelihoods,
            'final_log_likelihood': hmm.log_likelihoods[-1] if hmm.log_likelihoods else None
        }
    }
    
    filename = f"hmm_{dataset_type}_{series_id:03d}_{filename_base}_{timestamp}.pkl"
    filepath = os.path.join(checkpoint_dir, filename)
    
    with open(filepath, 'wb') as f:
        pickle.dump(checkpoint, f)
    
    return filepath


def train_multivariate_hmms_from_mat_files(
    adhd_dir,
    control_dir,
    checkpoint_dir='./hmm_checkpoints',
    n_states=3,
    n_gmm_components=3,
    max_iter=50,
    tol=1e-3,
    max_hours=11.5,
    param_tol=1e-3,
    flow_tol=1e-3,
    flow_norm='inf',
    min_param_iters=2,
    verbose_train=True
):
    """
    Full pipeline: load .mat files, train HMMs with early stopping.
    
    Parameters:
    -----------
    adhd_dir : str
        Directory containing ADHD .mat files
    control_dir : str
        Directory containing Control .mat files
    checkpoint_dir : str
        Directory to save checkpoints
    n_states : int
        Number of hidden states
    n_gmm_components : int
        Number of GMM components per state
    max_iter : int
        Maximum EM iterations
    tol : float
        Log-likelihood convergence tolerance
    max_hours : float
        Maximum training hours
    param_tol : float
        Parameter stationarity tolerance
    flow_tol : float
        Stationary flow tolerance
    flow_norm : str
        Norm for flow matrix ('inf' or 'fro')
    min_param_iters : int
        Minimum iterations before checking param convergence
    verbose_train : bool
        Print training progress
    
    Returns:
    --------
    tracker : HMMTrainingTracker
        Training tracker object
    failed_files : tuple
        (adhd_failed, control_failed)
    """
    print("=" * 80)
    print("   MULTIVARIATE EEG HMM TRAINING PIPELINE")
    print("=" * 80)

    time_manager = TimeManager(max_hours=max_hours)
    print(f" Training time limit: {max_hours} hours")

    adhd_data, control_data, adhd_files, control_files, adhd_failed, control_failed = load_all_eeg_data(
        adhd_dir, control_dir
    )
    
    if len(adhd_data) == 0 and len(control_data) == 0:
        print(" No data loaded successfully. Check file paths and formats.")
        return None, None

    tracker = HMMTrainingTracker(checkpoint_dir)

    total_planned = len(adhd_data) + len(control_data)
    tracker.progress['total_planned'] = total_planned
    if tracker.progress['start_time'] is None:
        tracker.progress['start_time'] = datetime.now().isoformat()
    tracker.save_progress()

    remaining_work = tracker.get_remaining_work(len(adhd_data), len(control_data))

    print(f"\n Starting/Resuming HMM Training")
    print(f"   Total HMMs to train: {total_planned}")
    print(f"   ADHD sequences: {len(adhd_data)}")
    print(f"   Control sequences: {len(control_data)}")
    if adhd_data:
        print(f"   Channels per subject: {adhd_data[0].shape[1]}")
    print(f"   HMM config: {n_states} states, {n_gmm_components} GMM components")
    print(f"   Max iterations: {max_iter}, LL tol: {tol}")
    print(f"   Param tol: {param_tol}, Flow tol: {flow_tol} ({flow_norm}-norm)")
    print(f"   Remaining work: {len(remaining_work)}")

    tracker.print_status(time_manager)

    for i, (dataset_type, series_id) in enumerate(remaining_work):
        if time_manager.should_stop():
            print(f"\n Time limit approaching — stopping to save results...")
            print(f"   Elapsed: {time_manager.format_elapsed_time()}")
            print(f"   Completed: {len(tracker.progress['completed_hmms'])}/{total_planned}")
            break

        try:
            print(f"\n Training {dataset_type} HMM #{series_id} ({i+1}/{len(remaining_work)})...")
            print(f"   Time elapsed: {time_manager.format_elapsed_time()}")

            if dataset_type == 'ADHD':
                if series_id >= len(adhd_data):
                    continue
                X_series = adhd_data[series_id]
                filename_base = adhd_files[series_id].replace('.mat', '') if series_id < len(adhd_files) else f"adhd_{series_id}"
            else:
                if series_id >= len(control_data):
                    continue
                X_series = control_data[series_id]
                filename_base = control_files[series_id].replace('.mat', '') if series_id < len(control_files) else f"control_{series_id}"

            print(f"     File: {filename_base}.mat")
            print(f"     Data shape: {X_series.shape} (samples={X_series.shape[0]}, channels={X_series.shape[1]})")

            hmm = MultivariateGaussianMixtureHMM(
                n_states=n_states,
                n_gmm_components=n_gmm_components,
                max_iter=max_iter,
                tol=tol,
                random_state=42 + series_id,
                param_tol=param_tol,
                min_param_iters=min_param_iters,
                early_stop_by_params=True,
                flow_tol=flow_tol,
                flow_norm=flow_norm,
                stop_on_flow=True
            )

            t0 = time.time()
            hmm.fit(X_series, verbose=verbose_train)
            training_time = time.time() - t0

            checkpoint_path = save_hmm_checkpoint(
                hmm, series_id, dataset_type, checkpoint_dir, filename_base
            )

            tracker.mark_completed(series_id, dataset_type, checkpoint_path, training_time)

            print(f"     Completed in {training_time:.1f}s")
            print(f"     Saved: {os.path.basename(checkpoint_path)}")
            print(f"     Converged: {hmm.converged}, Criterion: {hmm.convergence_criterion}")
            print(f"     Iterations: {hmm.final_iteration}")
            if hmm.log_likelihoods:
                print(f"     Final log-likelihood: {hmm.log_likelihoods[-1]:.6f}")

            del hmm

            if (len(tracker.progress['completed_hmms']) % 5 == 0):
                tracker.print_status(time_manager)
                gc.collect()

            if time_manager.time_for_checkpoint():
                print(f"     Periodic checkpoint — progress saved")
                tracker.print_status(time_manager)

        except Exception as e:
            error_msg = f"Error training {dataset_type} #{series_id}: {str(e)}"
            print(f"     {error_msg}")
            tracker.mark_failed(series_id, dataset_type, error_msg)
            continue

    print(f"\n Training Complete!")
    print(f"   Total time elapsed: {time_manager.format_elapsed_time()}")
    tracker.print_status(time_manager)

    return tracker, (adhd_failed, control_failed)


def collect_all_hmms_to_csv(checkpoint_dir, output_csv='multivariate_hmms_results.csv'):
    """
    Collect all trained HMMs into a final CSV dataset.
    
    Parameters:
    -----------
    checkpoint_dir : str
        Directory containing HMM checkpoints
    output_csv : str
        Path to output CSV file
    
    Returns:
    --------
    df : DataFrame
        Results DataFrame
    """
    tracker = HMMTrainingTracker(checkpoint_dir)
    
    print(f"\n Collecting all HMM results...")
    all_parameters = []
    
    for completed_item in tracker.progress['completed_hmms']:
        try:
            checkpoint_path = completed_item['checkpoint_path']
            
            with open(checkpoint_path, 'rb') as f:
                checkpoint = pickle.load(f)
            
            # Extract parameters with metadata
            params = checkpoint['parameters'].copy()
            params.update({
                'series_id': checkpoint['series_id'],
                'dataset_type': checkpoint['dataset_type'],
                'filename_base': checkpoint.get('filename_base', 'unknown'),
                'training_timestamp': checkpoint['timestamp'],
                'training_time_seconds': completed_item.get('training_time_seconds', np.nan),
                'convergence_criterion': checkpoint['convergence_info'].get('convergence_criterion', 'unknown')
            })
            
            all_parameters.append(params)
            
        except Exception as e:
            print(f"      Could not load {checkpoint_path}: {e}")
    
    # Create DataFrame and save
    if all_parameters:
        df = pd.DataFrame(all_parameters)
        df.to_csv(output_csv, index=False)
        
        print(f"     Saved {len(all_parameters)} HMM results to {output_csv}")
        print(f"     Columns: {len(df.columns)}")
        print(f"     ADHD HMMs: {len(df[df['dataset_type'] == 'ADHD'])}")
        print(f"     Control HMMs: {len(df[df['dataset_type'] == 'Control'])}")
        print(f"     Convergence rate: {df['converged'].mean():.1%}")
        print(f"     Avg training time: {df['training_time_seconds'].mean():.1f}s")
        
        # Convergence criterion breakdown
        print(f"\n Convergence Criteria Distribution:")
        print(df['convergence_criterion'].value_counts())
        
        # Additional statistics
        print(f"\n Final Statistics:")
        print(f"    Log-likelihood range: {df['final_log_likelihood'].min():.2f} to {df['final_log_likelihood'].max():.2f}")
        print(f"    Avg iterations: {df['final_iteration'].mean():.1f}")
        
        return df
    else:
        print("     No HMM results found")
        return None


if __name__ == "__main__":
    print(" Starting HMM Training Pipeline (.mat files with convergence tracking)")

    # ====================================================================
    # CONFIGURE PATHS
    # ====================================================================
    
    # Example 1: Synthetic EEG on Kaggle
    # adhd_dir = '/kaggle/input/synthetic-eeg/ADHD'
    # control_dir = '/kaggle/input/synthetic-eeg/Control'
    # checkpoint_dir = '/kaggle/working/hmm_checkpoints'
    # output_csv = '/kaggle/working/synthetic_hmms_results.csv'
    
    # Example 2: Real EEG on Kaggle
    # adhd_dir = '/kaggle/input/real-eeg/ADHD_prep'
    # control_dir = '/kaggle/input/real-eeg/Control_prep'
    # checkpoint_dir = '/kaggle/working/hmm_checkpoints'
    # output_csv = '/kaggle/working/real_hmms_results.csv'
    
    # Example 3: Local machine - Synthetic
    # adhd_dir = './data/synthetic/ADHD'
    # control_dir = './data/synthetic/Control'
    # checkpoint_dir = './hmm_checkpoints_synthetic'
    # output_csv = './synthetic_hmms_results.csv'
    
    # Example 4: Local machine - Real
    # adhd_dir = './data/real/ADHD_prep'
    # control_dir = './data/real/Control_prep'
    # checkpoint_dir = './hmm_checkpoints_real'
    # output_csv = './real_hmms_results.csv'
    
    # DEFAULT - CHANGE THE PATHS
    adhd_dir = '/kaggle/working/preprocessed/ADHD_prep'
    control_dir = '/kaggle/working/preprocessed/Control_prep'
    checkpoint_dir = '/kaggle/working/hmm_checkpoints'
    output_csv = '/kaggle/working/real_hmms_results_n3_f7.csv'
    
    # ====================================================================
    # RUN THE PIPELINE
    # ====================================================================
    
    tracker, failed_files = train_multivariate_hmms_from_mat_files(
        adhd_dir=adhd_dir,
        control_dir=control_dir,
        checkpoint_dir=checkpoint_dir,
        n_states=3,
        n_gmm_components=3,
        max_iter=30,
        tol=1e-3, #for Log-Likelihood
        max_hours= 12, # if kaggle: 12 hours; else (local machine) then sky is the limit.
        param_tol=1e-3,# for hmm parameters
        flow_tol=1e-3, # for stationary distribution
        flow_norm='inf',
        
        min_param_iters=2,
        verbose_train=True
    )

    # Collect results into CSV
    if tracker:
        results_df = collect_all_hmms_to_csv(
            checkpoint_dir=checkpoint_dir,
            output_csv=output_csv
        )
        if results_df is not None:
            print("\n FINAL SUMMARY:")
            print(f"   Total HMMs exported: {len(results_df)}")
            print(f"   Avg log-likelihood: {results_df['final_log_likelihood'].mean():.6f}")
            print(f"   Params per HMM: {len([c for c in results_df.columns if c.startswith(('pi_', 'A_', 'gmm_'))])}")
            print(f"   Converged rate: {results_df['converged'].mean():.1%}")
            

    print("\n Pipeline complete! Check output directory for:")
    print(f"  - {output_csv}")
    print(f"  - {checkpoint_dir}/ (pickle checkpoints)")